In [8]:
from turtle import done

import torch
import numpy as np
import pandas as pd


In [9]:
a = torch.tensor(2.0)
b = torch.tensor(3.0)
print(a + b)

tensor(5.)


In [15]:
file = "../data/data_with_features.csv"
data_df = pd.read_csv(file)

In [16]:
print(f"Dataset shape : {data_df.shape}\n")

Dataset shape : (100, 4)



In [19]:
print(data_df.head())

   distance_miles  time_of_day_hours  is_weekend  delivery_time_minutes
0            1.60               8.20           0                   7.22
1           13.09              16.80           1                  32.41
2            6.97               8.02           1                  17.47
3           10.66              16.07           0                  37.17
4           18.24              13.47           0                  38.36


In [20]:
sample_tensor = torch.tensor([
    [1.60,  8.20, 0,  7.22],
    [13.09, 16.80, 1, 32.41],
    [6.97,  8.02, 1, 17.47],
    [10.66, 16.07, 0, 37.17],
    [18.24, 13.47, 0, 38.36]
], dtype=torch.float32)

In [29]:
print(sample_tensor.dtype)

torch.float32


In [22]:
#simple linear model
import torch.nn as nn
import torch.optim as optim

In [39]:
#define the data
distances = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]])
times = torch.tensor([[7.0], [12.0], [17.0], [22.0], [27.0]])

#Forward pass
model = nn.Sequential(
    nn.Linear(1, 1)
)

#Loss
loss_function = nn.MSELoss()
#Optimizer
optimizer = optim.SGD(model.parameters(), lr = 0.01)

print("distance",distances.shape)
print("times ",times.shape)

#Training
for epoch in range(500):
    optimizer.zero_grad()
    outputs  = model(distances)

    loss = loss_function(outputs, times)
    loss.backward()
    optimizer.step()

distance torch.Size([5, 1])
times  torch.Size([5, 1])


In [43]:

#inference/ predfiction
with torch.no_grad():
    new_distance = torch.tensor([[8.0]],dtype=torch.float32)
    prediction = model(new_distance)

print("Predictions:")
print(prediction)


Predictions:
tensor([[41.8905]])


True


## Transforms - How should I preprocess this?

**raw example → transform → model-ready example**

# Dataset - "How do I organize and access my examples?"

# DataLoader - "How do I feed the Dataset into training?"


In [45]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader


In [47]:
#Temporary Transform
initial_transform = transforms.ToTensor()

#Loading the training dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform= initial_transform,
    download=True
)



100.0%
100.0%
100.0%
100.0%

torch.Size([1, 28, 28])
5


In [50]:
sum_pixels = 0.0
total_pixels = 0
for image, label in train_dataset:
    sum_pixels += image.sum() # sum of all pixels in the image
    total_pixels += image.numel() # number of pixels in the image
mean = sum_pixels / total_pixels
sum_squared_diff = 0.0

for image, _ in train_dataset:
    sum_squared_diff += ((image - mean) ** 2).sum()
variance = sum_squared_diff / total_pixels

std = torch.sqrt(variance)




In [51]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [52]:
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform= transform,
    download=True
)

In [53]:
test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    transform = transform,
    download=True
)

In [54]:
#Create Data Loader
train_loader = DataLoader(dataset=train_dataset,batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

- 784 = determined by the input image (28 × 28)
- 128 = chosen by us (hidden-layer size)
- 10 = determined by the problem (10 digit classes)
- -Original image [batch_size, 1, 28, 28]

In [55]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
        nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)#(128 hidden neurons and 10 class scores of the mnist
    )

    def forward(self,x):
        x = self.flatten(x)
        x = self.layers(x)
        return x


In [64]:
#Device and Optimizer
#Check for GOU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using {device} device')
#Initialize the model and move to device
model = MNISTClassifier().to(device)

#Loss function and optimizer
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Using cpu device


In [87]:
def train_model(model,train_loader, loss_function, optimizer,device = device):
    model.train() # tells pytorch to put this in a training
    running_loss = 0.0
    correct = 0
    total  = 0
    for batch_idx, (data,target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        #set the gradients to zero
        optimizer.zero_grad()
        #forward pass
        output = model(data)
        #calculate the loss
        loss = loss_function(output, target)
        #backprop
        loss.backward()
        #update the parameters
        optimizer.step()

        #Track progress
        running_loss += loss.item()
        _ , predicted = output.max(1) # containst the maximum value and its index
        total += target.size(0)
        correct += predicted.eq(target).sum().item()  #

        #Print every 100 batches
        if batch_idx % 100 == 0 and batch_idx > 0:
            avg_loss = running_loss / 100
            accuracy = 100.0 * correct / total
            print(f' [{total} / {len(train_loader.dataset)}]'
                f'Loss: {avg_loss:.3f} | '
                  f'Accuracy: {accuracy:.1f}%')
            running_loss = 0.0

        

In [88]:
train_model(model,train_loader,loss_function,optimizer)

 [3232 / 60000]Loss: 0.143 | Accuracy: 95.3%
 [6432 / 60000]Loss: 0.124 | Accuracy: 95.7%
 [9632 / 60000]Loss: 0.120 | Accuracy: 95.9%
 [12832 / 60000]Loss: 0.120 | Accuracy: 96.0%
 [16032 / 60000]Loss: 0.113 | Accuracy: 96.0%
 [19232 / 60000]Loss: 0.139 | Accuracy: 95.9%
 [22432 / 60000]Loss: 0.123 | Accuracy: 96.0%
 [25632 / 60000]Loss: 0.120 | Accuracy: 96.1%
 [28832 / 60000]Loss: 0.123 | Accuracy: 96.1%
 [32032 / 60000]Loss: 0.127 | Accuracy: 96.1%
 [35232 / 60000]Loss: 0.119 | Accuracy: 96.2%
 [38432 / 60000]Loss: 0.118 | Accuracy: 96.2%
 [41632 / 60000]Loss: 0.127 | Accuracy: 96.2%
 [44832 / 60000]Loss: 0.116 | Accuracy: 96.3%
 [48032 / 60000]Loss: 0.131 | Accuracy: 96.3%
 [51232 / 60000]Loss: 0.128 | Accuracy: 96.3%
 [54432 / 60000]Loss: 0.121 | Accuracy: 96.3%
 [57632 / 60000]Loss: 0.108 | Accuracy: 96.3%


In [89]:
def evaluate(model,test_loader,device):
    model.eval() # evaluate mode
    correct = total = 0

    with torch.no_grad():
        for inputs,targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            output = model(inputs)
            _, predicted = output.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    return 100.0 * correct / total

In [90]:
evaluate(model,test_loader,device)

96.77

In [91]:
#Putting everything together
num_epoch =10

for epoch in range(num_epoch):
    print(f'\nEpoch: {epoch+ 1}')
    train_model(model,train_loader,loss_function,optimizer,device)
    accuracy = evaluate(model,test_loader,device)
    print(f'Test Accuracy: {accuracy:.2f}%')


Epoch: 1
 [3232 / 60000]Loss: 0.120 | Accuracy: 96.7%
 [6432 / 60000]Loss: 0.121 | Accuracy: 96.4%
 [9632 / 60000]Loss: 0.094 | Accuracy: 96.7%
 [12832 / 60000]Loss: 0.098 | Accuracy: 96.7%
 [16032 / 60000]Loss: 0.098 | Accuracy: 96.7%
 [19232 / 60000]Loss: 0.086 | Accuracy: 96.8%
 [22432 / 60000]Loss: 0.101 | Accuracy: 96.8%
 [25632 / 60000]Loss: 0.106 | Accuracy: 96.8%
 [28832 / 60000]Loss: 0.095 | Accuracy: 96.8%
 [32032 / 60000]Loss: 0.099 | Accuracy: 96.8%
 [35232 / 60000]Loss: 0.104 | Accuracy: 96.8%
 [38432 / 60000]Loss: 0.121 | Accuracy: 96.8%
 [41632 / 60000]Loss: 0.089 | Accuracy: 96.9%
 [44832 / 60000]Loss: 0.093 | Accuracy: 96.9%
 [48032 / 60000]Loss: 0.114 | Accuracy: 96.8%
 [51232 / 60000]Loss: 0.093 | Accuracy: 96.9%
 [54432 / 60000]Loss: 0.128 | Accuracy: 96.8%
 [57632 / 60000]Loss: 0.094 | Accuracy: 96.8%
Accuracy: 96.57%

Epoch: 2
 [3232 / 60000]Loss: 0.084 | Accuracy: 97.6%
 [6432 / 60000]Loss: 0.082 | Accuracy: 97.4%
 [9632 / 60000]Loss: 0.080 | Accuracy: 97.5%
 [1